In [0]:
from pyspark.sql.types import StringType
import pyspark.sql.functions as F

In [0]:
df = spark.read.table("workspace.bronze.crm_sales_details")
display(df.printSchema())
display(df.show())

In [0]:
# trimming
for col in df.schema.fields:
    if isinstance(col.dataType, StringType):
        df = df.withColumn(col.name, F.trim(col.name))

df.show()

In [0]:
# rename
mapping_names = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in mapping_names.items():
    df = df.withColumnRenamed(old_name, new_name)

df.printSchema()

In [0]:
# cleaning date fields
df = df.withColumn(
    "order_date", 
    F.when(F.col("order_date") < 10000000, None)
    .otherwise(F.to_date(F.col("order_date").cast("string"), "yyyyMMdd"))
).withColumn(
    "ship_date", 
    F.when(F.col("ship_date") < 10000000, None)
    .otherwise(F.to_date(F.col("ship_date").cast("string"), "yyyyMMdd"))
).withColumn(
    "due_date", 
    F.when(F.col("due_date") < 10000000, None)
    .otherwise(F.to_date(F.col("due_date").cast("string"), "yyyyMMdd"))
)

In [0]:
# handle incorrect sales/price/quantity
df = df.withColumn(
    "price", 
    F.when(F.col("price") < 0, -1 * F.col("price"))
    .otherwise(F.col("price"))
).withColumn(
    "is_invalid_sales",
    F.when(F.col("sales") < 0, True)
    .when(F.col("sales") != F.col("price"), True)
    .otherwise(False)
)

In [0]:
df.show()

In [0]:
# writing to delta
df.write.mode("overwrite").saveAsTable("workspace.silver.crm_sales_details")